# QuantJourney SDK - Institutional Crowding and 13F Flow Signals

This notebook demonstrates a QuantJourney SDK workflow that uses SEC/FMP institutional ownership calls with returns data to build crowding, concentration and flow proxies.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbols = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META']
ownership = {symbol: qj.sec.get_institutional_holdings(symbol=symbol) for symbol in symbols[:3]}
fmp_holders = {symbol: qj.fmp.get_institutional_holders(symbol=symbol) for symbol in symbols[:3]}
prices, volumes = price_panel(symbols)
ret = returns(prices)


In [ ]:
rows = []
for symbol, payload in fmp_holders.items():
    holder_rows = as_rows(payload)
    values = []
    for item in holder_rows:
        value = item.get('value') or item.get('marketValue') or item.get('shares')
        values.append(pd.to_numeric(value, errors='coerce'))
    values = pd.Series(values).dropna()
    rows.append({'symbol': symbol, 'holder_count': len(holder_rows), 'top10_concentration': values.nlargest(10).sum() / values.sum() if values.sum() else np.nan})
crowding = pd.DataFrame(rows).set_index('symbol')
crowding['momentum_126d'] = prices.pct_change(126).iloc[-1].reindex(crowding.index)
display(crowding)
crowding[['holder_count', 'top10_concentration', 'momentum_126d']].plot(kind='bar', subplots=True, layout=(1, 3), figsize=(15, 4), title='Crowding diagnostics')
plt.tight_layout()
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.